In [1]:
import pandas as pd

In [2]:
#  Load the files
orders= pd.read_csv("olist_orders_dataset.csv")

In [3]:
orders.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00


In [4]:
print(orders['order_status'].value_counts())

order_status
delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2
Name: count, dtype: int64


In [5]:
print(orders.isnull().sum())

order_id                            0
customer_id                         0
order_status                        0
order_purchase_timestamp            0
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
order_estimated_delivery_date       0
dtype: int64


In [6]:
#  Load the files
items  = pd.read_csv("olist_order_items_dataset.csv")

In [7]:
items.head()

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.90,13.29
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.90,19.93
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.00,17.87
3,00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-15 10:10:18,12.99,12.79
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-13 13:57:51,199.90,18.14


In [8]:
merged_data = pd.merge(orders, items, on='order_id', how='left')

In [9]:
# Calculate the 'Financial Struggle'
# We filter for 'canceled' and then sum the 'price' column
cancelled_value = merged_data[merged_data['order_status']== 'canceled']['price'].sum()

In [10]:
print(f"Total Revenue Lost to Canceled Orders: ${cancelled_value:,.2f}")

Total Revenue Lost to Canceled Orders: $95,235.27


In [11]:
#  Check for 'Hidden' Losses (Unavailable orders)
unavailable_value = merged_data[merged_data['order_status'] == 'unavailable']['price'].sum()
print(f"Total Revenue Lost to Unavailable Orders: ${unavailable_value:,.2f}")

Total Revenue Lost to Unavailable Orders: $2,007.69


In [13]:
from google.colab import files
uploaded=files.upload()

Saving olist_products_dataset.csv to olist_products_dataset (1).csv


In [14]:
#  Load the Product details
products = pd.read_csv('olist_products_dataset.csv')

In [15]:
# Merge our existing data with the products table
# We use 'product_id' as the glue this time
final_df = pd.merge(merged_data, products, on='product_id', how='left')


In [16]:

# Find the "Problem Categories"
# Let's see which categories have the most 'canceled' orders
problem_categories = final_df[final_df['order_status'] == 'canceled']['product_category_name'].value_counts()

In [17]:
print("--- Top 5 Categories with Canceled Orders ---")
print(problem_categories.head(5))

--- Top 5 Categories with Canceled Orders ---
product_category_name
esporte_lazer             51
utilidades_domesticas     49
informatica_acessorios    46
moveis_decoracao          36
beleza_saude              36
Name: count, dtype: int64


In [18]:
from google.colab import files
uploaded=files.upload()

Saving product_category_name_translation.csv to product_category_name_translation (2).csv


In [19]:
# 1. Load the translation file
translation = pd.read_csv('product_category_name_translation.csv')

# 2. Merge it into your final dataframe
# This adds a new column called 'product_category_name_english'
final_df = pd.merge(final_df, translation, on='product_category_name', how='left')

# 3. Now calculate the Canceled counts using the ENGLISH names
english_problem_cats = final_df[final_df['order_status'] == 'canceled']['product_category_name_english'].value_counts()

print("--- Top 5 Problem Categories (English) ---")
print(english_problem_cats.head(5))


--- Top 5 Problem Categories (English) ---
product_category_name_english
sports_leisure           51
housewares               49
computers_accessories    46
furniture_decor          36
health_beauty            36
Name: count, dtype: int64


In [20]:
# 1. Count Total Orders per Category
total_per_cat = final_df['product_category_name'].value_counts()

# 2. Count Canceled Orders per Category
canceled_per_cat = final_df[final_df['order_status'] == 'canceled']['product_category_name'].value_counts()

# 3. Calculate the Percentage
cancel_rate = (canceled_per_cat / total_per_cat) * 100

# 4. Sort to see the highest percentage first
print("--- Categories with Highest CANCELLATION RATE (%) ---")
print(cancel_rate.sort_values(ascending=False).head(10))


--- Categories with Highest CANCELLATION RATE (%) ---
product_category_name
pc_gamer                                         11.111111
portateis_cozinha_e_preparadores_de_alimentos     6.666667
dvds_blu_ray                                      3.125000
construcao_ferramentas_seguranca                  2.577320
fraldas_higiene                                   2.564103
construcao_ferramentas_jardim                     1.680672
instrumentos_musicais                             1.617647
livros_interesse_geral                            1.265823
eletrodomesticos_2                                1.260504
eletroportateis                                   1.178203
Name: count, dtype: float64


In [21]:
from google.colab import files
uploaded=files.upload()


Saving olist_order_reviews_dataset.csv to olist_order_reviews_dataset (1).csv


In [22]:
# 1. Load the Reviews
reviews = pd.read_csv('olist_order_reviews_dataset.csv')

# 2. Merge reviews with our final_df
# Note: One order can sometimes have multiple reviews, so we join on order_id
final_with_reviews = pd.merge(final_df, reviews, on='order_id', how='left')

# 3. The "Smoking Gun" Analysis:
# Let's compare the Average Review Score for 'delivered' vs 'canceled' orders
avg_scores = final_with_reviews.groupby('order_status')['review_score'].mean()

print("--- Average Review Score by Order Status ---")
print(avg_scores)


--- Average Review Score by Order Status ---
order_status
approved       2.000000
canceled       1.793054
created        2.333333
delivered      4.081309
invoiced       1.647222
processing     1.340909
shipped        1.996400
unavailable    1.530100
Name: review_score, dtype: float64


In [23]:
# 1. Create a summary of every product
product_summary = final_with_reviews.groupby('product_id').agg({
    'order_id': 'count',           # Total times ordered
    'price': 'sum',                # Total revenue
    'review_score': 'mean',        # Average happiness
    'order_status': lambda x: (x == 'canceled').sum() # Total cancels
})

# 2. Rename columns to make them clear
product_summary.columns = ['total_orders', 'total_revenue', 'avg_review', 'cancel_count']

# 3. Filter for products with at least 5 orders (to avoid random one-offs)
# and sort by the most cancels
hit_list = product_summary[product_summary['total_orders'] >= 5].sort_values(by='cancel_count', ascending=False)

print("--- THE HIT LIST: Top 5 Revenue Killers ---")
print(hit_list.head(5))


--- THE HIT LIST: Top 5 Revenue Killers ---
                                  total_orders  total_revenue  avg_review  \
product_id                                                                  
8397dc503d1a0c2ac7422701884de5a6             6        1854.00    1.000000   
5c3eaf54e8ee5d5378765ff16df7640b             6          77.94    1.000000   
75f3ef6a5cb0f2d5aeef15925f0ccf69             6        1639.45    1.666667   
3ea32f63a6aaf8d467e543dedf434ee7             5         267.00    5.000000   
ed08ea04c92f5f434c2362f7310fb328             5          27.65    1.000000   

                                  cancel_count  
product_id                                      
8397dc503d1a0c2ac7422701884de5a6             6  
5c3eaf54e8ee5d5378765ff16df7640b             6  
75f3ef6a5cb0f2d5aeef15925f0ccf69             5  
3ea32f63a6aaf8d467e543dedf434ee7             5  
ed08ea04c92f5f434c2362f7310fb328             5  


In [24]:
# --- THE LOAD STEP ---
# This saves your final, cleaned, and merged data into a single 'Master File'
final_with_reviews.to_csv('cleaned_ecommerce_master.csv', index=False)

print("Pipeline Complete! Cleaned master file saved as 'cleaned_ecommerce_master.csv'")


Pipeline Complete! Cleaned master file saved as 'cleaned_ecommerce_master.csv'
